# Проверка метаполей и идемпотентности расчётов

Ноутбук анализирует метаполя результирующих таблиц `user_scd2` и `campaign_recipients` для трёх scheduled-интервалов 23–25 января 2026 года.

Проверяются заполненность обязательных метаполей, корректность границ интервалов, согласованность идентификатора запуска между двумя задачами одного DAG run и время формирования результатов.

В заключительной части выполняется сравнение сохранённых до повторного расчёта партиций с результатами после повторного выполнения задач за 23 января. Для этой проверки используются резервные копии всех шести партиций обеих таблиц за три даты.


In [1]:
from jobs.common import create_spark_session, dataset_name, s3_uri
from pyspark.sql import functions as F

spark = create_spark_session("hw23-notebook-metadata-check")
dataset = dataset_name()
dates = ["2026-01-23", "2026-01-24", "2026-01-25"]
META_COLUMNS = [
    "_dag_id", "_run_id", "_task_id",
    "_data_interval_start", "_data_interval_end", "_loaded_at",
]


26/11/20 20:50:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Сводка метаполей

Для каждой партиции обеих таблиц группируются строки по `_dag_id`, `_run_id`, `_task_id` и границам интервала обработки. Выводятся количество строк, минимальное и максимальное значения `_loaded_at`.

Сводка позволяет проверить, что строки каждой партиции относятся к ожидаемому запуску Airflow, а время загрузки зафиксировано в пределах одной попытки задачи.


In [2]:
tables = {}
for date in dates:
    tables[("scd2", date)] = spark.read.parquet(
        s3_uri(f"results/{dataset}/user_scd2/snapshot_date={date}")
    )
    tables[("recipients", date)] = spark.read.parquet(
        s3_uri(f"results/{dataset}/campaign_recipients/calculation_date={date}")
    )

for (table_name, date), df in tables.items():
    print(f"\n{table_name} / {date}")
    df.groupBy(*META_COLUMNS[:-1]).agg(
        F.count("*").alias("row_count"),
        F.min("_loaded_at").alias("min_loaded_at"),
        F.max("_loaded_at").alias("max_loaded_at"),
    ).show(truncate=False)


26/11/20 20:50:07 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties



scd2 / 2026-01-23
+------------------------+------------------------------------+---------------+--------------------+-------------------+---------+-------------------------+-------------------------+
|_dag_id                 |_run_id                             |_task_id       |_data_interval_start|_data_interval_end |row_count|min_loaded_at            |max_loaded_at            |
+------------------------+------------------------------------+---------------+--------------------+-------------------+---------+-------------------------+-------------------------+
|user_campaign_scd2_daily|scheduled__2026-01-23T00:00:00+00:00|build_user_scd2|2026-01-23 00:00:00 |2026-01-24 00:00:00|961      |2026-11-20 20:47:57.08591|2026-11-20 20:47:57.08591|
+------------------------+------------------------------------+---------------+--------------------+-------------------+---------+-------------------------+-------------------------+


recipients / 2026-01-23
+------------------------+--------------

## Проверка заполненности метаполей и интервалов

Для каждой результирующей партиции проверяется отсутствие `NULL` во всех шести обязательных метаполях. Дополнительно проверяется, что продолжительность интервала обработки составляет ровно одни сутки.

Проверка применяется к обеим таблицам и ко всем трём датам расчёта.


In [3]:
for (table_name, date), df in tables.items():
    null_condition = None
    for column in META_COLUMNS:
        condition = F.col(column).isNull()
        null_condition = condition if null_condition is None else (null_condition | condition)
    null_count = df.filter(null_condition).count()
    invalid_interval = df.filter(
        (F.col("_data_interval_end") - F.col("_data_interval_start") != F.expr("INTERVAL 1 DAY"))
    ).count()
    print(table_name, date, "NULL metadata rows:", null_count, "invalid intervals:", invalid_interval)
    assert null_count == 0
    assert invalid_interval == 0


scd2 2026-01-23 NULL metadata rows: 0 invalid intervals: 0
recipients 2026-01-23 NULL metadata rows: 0 invalid intervals: 0
scd2 2026-01-24 NULL metadata rows: 0 invalid intervals: 0
recipients 2026-01-24 NULL metadata rows: 0 invalid intervals: 0
scd2 2026-01-25 NULL metadata rows: 0 invalid intervals: 0
recipients 2026-01-25 NULL metadata rows: 0 invalid intervals: 0


## Согласованность двух задач одного DAG run

Для каждой даты сопоставляются значения `_dag_id`, `_run_id`, `_data_interval_start` и `_data_interval_end` в таблицах `user_scd2` и `campaign_recipients`.

Поскольку обе задачи выполняются в рамках одного DAG run и обрабатывают один интервал, соответствующие значения должны совпадать. Сравнение множеств выполняется в обе стороны.


In [4]:
for date in dates:
    left = tables[("scd2", date)].select("_dag_id", "_run_id", "_data_interval_start", "_data_interval_end").distinct()
    right = tables[("recipients", date)].select("_dag_id", "_run_id", "_data_interval_start", "_data_interval_end").distinct()
    print("Дата:", date)
    print("SCD2 metadata:")
    left.show(truncate=False)
    print("Recipients metadata:")
    right.show(truncate=False)
    # При непустых таблицах множества run_id/интервалов должны совпасть.
    if left.limit(1).count() and right.limit(1).count():
        assert left.exceptAll(right).limit(1).count() == 0
        assert right.exceptAll(left).limit(1).count() == 0


Дата: 2026-01-23
SCD2 metadata:
+------------------------+------------------------------------+--------------------+-------------------+
|_dag_id                 |_run_id                             |_data_interval_start|_data_interval_end |
+------------------------+------------------------------------+--------------------+-------------------+
|user_campaign_scd2_daily|scheduled__2026-01-23T00:00:00+00:00|2026-01-23 00:00:00 |2026-01-24 00:00:00|
+------------------------+------------------------------------+--------------------+-------------------+

Recipients metadata:
+------------------------+------------------------------------+--------------------+-------------------+
|_dag_id                 |_run_id                             |_data_interval_start|_data_interval_end |
+------------------------+------------------------------------+--------------------+-------------------+
|user_campaign_scd2_daily|scheduled__2026-01-23T00:00:00+00:00|2026-01-23 00:00:00 |2026-01-24 00:00:00|
+

## Проверка идемпотентности повторного расчёта

Проверка состоит из резервного копирования и последующего сравнения результатов.

Перед повторным выполнением задач сохраняются копии всех шести партиций: `user_scd2` и `campaign_recipients` за 23, 24 и 25 января 2026 года. После резервного копирования обе задачи scheduled-запуска за 23 января повторно выполняются через Airflow Clear.

При сравнении результатов за 23 января исключаются только шесть метаполей; бизнес-поля и поле даты должны полностью совпасть. Для 24 и 25 января сравниваются все столбцы, включая метаполя, поскольку повторный запуск более раннего интервала не должен изменять последующие партиции.

Сравнение выполняется операцией `exceptAll` в обоих направлениях, что проверяет значения и кратность строк. Резервные копии сохраняются в отдельном S3-префиксе `results/{dataset}/_rerun_backup_2026-01-23/`.


In [6]:
# Сначала выполнить с "backup" ДО Clear в Airflow.
# После повторного успешного выполнения 23.01 изменить на "verify" и выполнить ячейку ещё раз.
RERUN_CHECK_MODE = "backup"  # допустимые значения: "backup" или "verify"
RERUN_DATE = "2026-01-23"
BACKUP_ROOT = s3_uri(f"results/{dataset}/_rerun_backup_{RERUN_DATE}")

TABLES_TO_CHECK = {
    "scd2": {
        "relative_path": "user_scd2/snapshot_date=",
        "date_column": "snapshot_date",
    },
    "recipients": {
        "relative_path": "campaign_recipients/calculation_date=",
        "date_column": "calculation_date",
    },
}
ALL_DATES = ["2026-01-23", "2026-01-24", "2026-01-25"]
META_COLUMNS = [
    "_dag_id", "_run_id", "_task_id", "_data_interval_start",
    "_data_interval_end", "_loaded_at",
]

if RERUN_CHECK_MODE == "backup":
    print("Создание резервных копий шести партиций до повторного запуска.")
    for table_name, spec in TABLES_TO_CHECK.items():
        for date_text in ALL_DATES:
            source_path = s3_uri(
                f"results/{dataset}/{spec['relative_path']}{date_text}"
            )
            backup_path = (
                f"{BACKUP_ROOT}/{table_name}/{spec['date_column']}={date_text}"
            )
            # errorifexists защищает исходную резервную копию от случайной перезаписи.
            source_df = spark.read.parquet(source_path)
            source_df.write.mode("errorifexists").parquet(backup_path)
            backup_count = spark.read.parquet(backup_path).count()
            assert backup_count == source_df.count()
            print(f"{table_name} {date_text}: сохранено строк {backup_count}")
    print("Резервные копии готовы. Теперь выполните Clear только для двух задач запуска 23.01.")

elif RERUN_CHECK_MODE == "verify":
    comparison_results = []
    for table_name, spec in TABLES_TO_CHECK.items():
        for date_text in ALL_DATES:
            backup_path = (
                f"{BACKUP_ROOT}/{table_name}/{spec['date_column']}={date_text}"
            )
            current_path = s3_uri(
                f"results/{dataset}/{spec['relative_path']}{date_text}"
            )
            before_df = spark.read.parquet(backup_path)
            after_df = spark.read.parquet(current_path)

            assert set(before_df.columns) == set(after_df.columns), (
                f"Изменился набор столбцов: {table_name} {date_text}"
            )
            # 23 января: исключаем только шесть метаполей. 24/25 января
            # сравниваем все поля, включая метаполя, как требует задание.
            compare_columns = [c for c in before_df.columns if c not in META_COLUMNS] \
                if date_text == RERUN_DATE else before_df.columns
            before_cmp = before_df.select(*compare_columns)
            after_cmp = after_df.select(*compare_columns)

            only_before = before_cmp.exceptAll(after_cmp).count()
            only_after = after_cmp.exceptAll(before_cmp).count()
            before_count = before_cmp.count()
            after_count = after_cmp.count()
            result = {
                "table": table_name,
                "date": date_text,
                "compared_columns": len(compare_columns),
                "rows_before": before_count,
                "rows_after": after_count,
                "only_before": only_before,
                "only_after": only_after,
            }
            comparison_results.append(result)
            print(result)
            assert only_before == 0 and only_after == 0, (
                f"Найдены различия: {table_name} {date_text}: "
                f"only_before={only_before}, only_after={only_after}"
            )

    print("Идемпотентность подтверждена: все шесть сравнений прошли.")
else:
    raise ValueError("RERUN_CHECK_MODE должен быть 'backup' или 'verify'")

Создание резервных копий шести партиций до повторного запуска.
scd2 2026-01-23: сохранено строк 961
scd2 2026-01-24: сохранено строк 995
scd2 2026-01-25: сохранено строк 1036
recipients 2026-01-23: сохранено строк 53
recipients 2026-01-24: сохранено строк 25
recipients 2026-01-25: сохранено строк 22
Резервные копии готовы. Теперь выполните Clear только для двух задач запуска 23.01.
